In [2]:
import cv2
import numpy as np
import os

def create_mask(image_path, target_rgb, output_dir):
    # 读取图像
    image = cv2.imread(image_path)
    if image is None:
        print(f"Failed to load image: {image_path}")
        return

    # 创建一个全黑的掩码图像，与原图同尺寸和通道数
    mask = np.zeros_like(image)

    # 将目标RGB值转换为BGR格式，因为OpenCV使用BGR格式
    target_bgr = (target_rgb[2], target_rgb[1], target_rgb[0])

    # 找到所有目标BGR值的像素位置
    target_pixels = np.all(image == target_bgr, axis=-1)  # 确保所有通道都匹配

    # 将这些位置设为白色
    mask[target_pixels] = [255, 255, 255]

    # 构建输出文件路径
    filename = os.path.basename(image_path)
    output_path = os.path.join(output_dir, f"mask_{filename}")

    # 保存掩码图像
    cv2.imwrite(output_path, mask)
    print(f"Mask saved to: {output_path}")

def process_images_in_directory(image_dir, target_rgb, output_dir):
    # 确保输出目录存在
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    # 遍历目录中的所有文件
    for filename in os.listdir(image_dir):
        if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
            image_path = os.path.join(image_dir, filename)
            create_mask(image_path, target_rgb, output_dir)

In [3]:
# 示例：处理目录下的图像
image_dir = 'annotions-val/stuff_val2017_pixelmaps'  # 替换为你的图像文件夹路径
output_dir = 'val-mask'  # 替换为你的输出文件夹路径
target_rgb = (255, 214, 0)  # 目标RGB值
process_images_in_directory(image_dir, target_rgb, output_dir)

Mask saved to: val-mask\mask_000000000139.png
Mask saved to: val-mask\mask_000000000285.png
Mask saved to: val-mask\mask_000000000632.png
Mask saved to: val-mask\mask_000000000724.png
Mask saved to: val-mask\mask_000000000776.png
Mask saved to: val-mask\mask_000000000785.png
Mask saved to: val-mask\mask_000000000802.png
Mask saved to: val-mask\mask_000000000872.png
Mask saved to: val-mask\mask_000000000885.png
Mask saved to: val-mask\mask_000000001000.png
Mask saved to: val-mask\mask_000000001268.png
Mask saved to: val-mask\mask_000000001296.png
Mask saved to: val-mask\mask_000000001353.png
Mask saved to: val-mask\mask_000000001425.png
Mask saved to: val-mask\mask_000000001490.png
Mask saved to: val-mask\mask_000000001503.png
Mask saved to: val-mask\mask_000000001532.png
Mask saved to: val-mask\mask_000000001584.png
Mask saved to: val-mask\mask_000000001675.png
Mask saved to: val-mask\mask_000000001761.png
Mask saved to: val-mask\mask_000000001818.png
Mask saved to: val-mask\mask_00000

In [17]:
import cv2
import numpy as np
import os

import cv2
import numpy as np
import os

def apply_mask_to_image(image_path, mask_path, output_path):
    # 读取原图和掩码图像
    image = cv2.imread(image_path)
    mask = cv2.imread(mask_path, cv2.IMREAD_COLOR)  # 读取三通道掩码

    if image is None or mask is None:
        print(f"Failed to load image or mask at {image_path} or {mask_path}")
        return

    # 确保掩码和原图大小相同
    if mask.shape[:2] != image.shape[:2]:
        print("Mask and image must have the same dimensions.")
        return

    # 将掩码转换为灰度图，以便更容易地处理白色部分
    gray_mask = cv2.cvtColor(mask, cv2.COLOR_BGR2GRAY)

    # 找到掩码中所有白色部分（像素值为255）
    _, white_mask = cv2.threshold(gray_mask, 127, 255, cv2.THRESH_BINARY)

    # 将掩码转换回三通道，以便与原图相乘
    white_mask_3channel = cv2.cvtColor(white_mask, cv2.COLOR_GRAY2BGR)

    # 将原图中掩码为白色的部分变为白色
    for c in range(3):
        image[:, :, c][white_mask_3channel[:, :, 0] == 255] = 255

    # 保存或显示修改后的图像
    cv2.imwrite(output_path, image)
    print(f"Modified image saved to: {output_path}")

def process_images_in_directory(image_dir, mask_dir, output_dir):
    # 确保输出目录存在
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    # 遍历目录中的所有文件
    for filename in os.listdir(image_dir):
        if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
            image_path = os.path.join(image_dir, filename)
            # 构建掩码文件名，保留原始文件扩展名
            mask_filename = 'mask_' + os.path.splitext(filename)[0] + '.png'
            mask_path = os.path.join(mask_dir, mask_filename)

            if os.path.exists(mask_path):
                output_filename = 'masked_' + filename
                output_path = os.path.join(output_dir, output_filename)
                apply_mask_to_image(image_path, mask_path, output_path)
            else:
                print(f"Mask not found for {filename}")

# 示例：处理目录下的图像
image_dir = 'val2017'  # 替换为你的原图文件夹路径
mask_dir = 'val-mask'    # 替换为你的掩码文件夹路径
output_dir = 'val-masked_image'      # 替换为你的输出文件夹路径
process_images_in_directory(image_dir, mask_dir, output_dir)

Modified image saved to: val-masked_image\masked_000000000139.jpg
Modified image saved to: val-masked_image\masked_000000000285.jpg
Modified image saved to: val-masked_image\masked_000000000632.jpg
Modified image saved to: val-masked_image\masked_000000000724.jpg
Modified image saved to: val-masked_image\masked_000000000776.jpg
Modified image saved to: val-masked_image\masked_000000000785.jpg
Modified image saved to: val-masked_image\masked_000000000802.jpg
Modified image saved to: val-masked_image\masked_000000000872.jpg
Modified image saved to: val-masked_image\masked_000000000885.jpg
Modified image saved to: val-masked_image\masked_000000001000.jpg
Modified image saved to: val-masked_image\masked_000000001268.jpg
Modified image saved to: val-masked_image\masked_000000001296.jpg
Modified image saved to: val-masked_image\masked_000000001353.jpg
Modified image saved to: val-masked_image\masked_000000001425.jpg
Modified image saved to: val-masked_image\masked_000000001490.jpg
Modified i